In [0]:

# Inspect order/payment status combinations and currencies

result = spark.sql("""
    SELECT
        o.order_status,
        t.txn_status,
        t.currency,
        COUNT(*) AS transaction_count,
        SUM(t.amount) AS total_payment_amount

    FROM workspace.ecom_poc.silver_transactions t

    INNER JOIN workspace.ecom_poc.silver_orders o
        ON t.order_id = o.order_id

    GROUP BY
        o.order_status,
        t.txn_status,
        t.currency

    ORDER BY
        o.order_status,
        t.txn_status,
        t.currency
""")

display(result)


In [0]:

spark.sql("""
CREATE OR REPLACE VIEW workspace.ecom_poc.gold_order_revenue_base AS

WITH payments_per_order AS (

    SELECT
        order_id,
        currency,
        SUM(amount) AS successful_payment_amount

    FROM workspace.ecom_poc.silver_transactions

    WHERE txn_status = 'SUCCESS'

    GROUP BY order_id, currency
),

orders_with_currency AS (

    SELECT
        order_id,
        customer_id,
        order_date,
        category,
        country,
        order_status,

        CASE country
            WHEN 'Bangladesh' THEN 'BDT'
            WHEN 'India' THEN 'INR'
            WHEN 'USA' THEN 'USD'
            WHEN 'UK' THEN 'GBP'
        END AS currency,

        CAST(
            quantity * unit_price AS DECIMAL(28,2)
        ) AS order_value

    FROM workspace.ecom_poc.silver_orders
)

SELECT
    o.*,

    COALESCE(
        p.successful_payment_amount,
        CAST(0 AS DECIMAL(28,2))
    ) AS successful_payment_amount,

    CASE
        WHEN o.order_status = 'COMPLETED'
        THEN LEAST(
            o.order_value,
            COALESCE(
                p.successful_payment_amount,
                CAST(0 AS DECIMAL(28,2))
            )
        )
        ELSE CAST(0 AS DECIMAL(28,2))
    END AS revenue

FROM orders_with_currency o

LEFT JOIN payments_per_order p
    ON o.order_id = p.order_id
    AND o.currency = p.currency
""")

print("Gold revenue foundation view created!")


In [0]:

from pyspark.sql import functions as F

revenue_df = spark.table(
    "workspace.ecom_poc.gold_order_revenue_base"
)

result = revenue_df.agg(
    F.count("*").alias("total_orders"),
    F.countDistinct("order_id").alias("unique_orders"),
    F.sum(
        F.when(
            F.col("revenue") < 0,
            1
        ).otherwise(0)
    ).alias("negative_revenue"),
    F.sum(
        F.when(
            F.col("revenue") > F.col("order_value"),
            1
        ).otherwise(0)
    ).alias("overpaid_revenue"),
    F.sum(
        F.when(
            F.col("currency").isNull(),
            1
        ).otherwise(0)
    ).alias("missing_currency")
).first()

print("Total orders:", result["total_orders"])
print("Unique orders:", result["unique_orders"])
print("Negative revenue records:", result["negative_revenue"])
print("Revenue above order value:", result["overpaid_revenue"])
print("Missing currency:", result["missing_currency"])

assert result["total_orders"] == 984_082
assert result["unique_orders"] == 984_082
assert result["negative_revenue"] == 0
assert result["overpaid_revenue"] == 0
assert result["missing_currency"] == 0

print("REVENUE FOUNDATION VERIFICATION: PASSED")


In [0]:
display(
    spark.sql("""
        SELECT
            order_id,
            order_status,
            currency,
            order_value,
            successful_payment_amount,
            revenue
        FROM workspace.ecom_poc.gold_order_revenue_base
        WHERE successful_payment_amount > 0
        LIMIT 15
    """)
)

In [0]:

spark.sql("""
CREATE OR REPLACE TABLE workspace.ecom_poc.gold_daily_sales
USING DELTA
AS
SELECT
    order_date AS sales_date,
    currency,

    COUNT(*) AS total_orders,

    COUNT_IF(
        order_status = 'COMPLETED'
    ) AS completed_orders,

    COUNT_IF(
        order_status = 'CANCELLED'
    ) AS cancelled_orders,

    CAST(
        SUM(revenue) AS DECIMAL(28,2)
    ) AS total_revenue,

    ROUND(
        100.0 * COUNT_IF(order_status = 'CANCELLED')
        / COUNT(*),
        2
    ) AS cancellation_rate_pct

FROM workspace.ecom_poc.gold_order_revenue_base

GROUP BY
    order_date,
    currency
""")

print("Gold daily sales table created!")


In [0]:

from pyspark.sql import functions as F

daily_sales = spark.table(
    "workspace.ecom_poc.gold_daily_sales"
)

# Check row count and order accounting
result = daily_sales.agg(
    F.count("*").alias("gold_rows"),
    F.sum("total_orders").alias("total_orders"),
    F.sum("completed_orders").alias("completed_orders"),
    F.sum("cancelled_orders").alias("cancelled_orders"),
    F.min("cancellation_rate_pct").alias("min_rate"),
    F.max("cancellation_rate_pct").alias("max_rate")
).first()

# Each date/currency combination must appear once
duplicate_groups = (
    daily_sales
    .groupBy("sales_date", "currency")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Gold daily sales rows:", result["gold_rows"])
print("Orders represented:", result["total_orders"])
print("Completed orders:", result["completed_orders"])
print("Cancelled orders:", result["cancelled_orders"])
print("Minimum cancellation rate:", result["min_rate"])
print("Maximum cancellation rate:", result["max_rate"])
print("Duplicate date/currency groups:", duplicate_groups)

assert result["gold_rows"] > 0
assert result["total_orders"] == 984_082
assert duplicate_groups == 0
assert result["min_rate"] >= 0
assert result["max_rate"] <= 100

print("GOLD DAILY SALES VERIFICATION: PASSED")


In [0]:

display(
    spark.sql("""
        SELECT *
        FROM workspace.ecom_poc.gold_daily_sales
        ORDER BY sales_date, currency
        LIMIT 20
    """)
)


In [0]:

spark.sql("""
CREATE OR REPLACE TABLE workspace.ecom_poc.gold_revenue_breakdown
USING DELTA
AS
SELECT
    order_date AS sales_date,
    category,
    country,
    currency,

    COUNT(*) AS total_orders,

    COUNT_IF(
        order_status = 'COMPLETED'
    ) AS completed_orders,

    CAST(
        SUM(revenue) AS DECIMAL(28,2)
    ) AS total_revenue

FROM workspace.ecom_poc.gold_order_revenue_base

GROUP BY
    order_date,
    category,
    country,
    currency
""")

print("Gold revenue breakdown table created!")


In [0]:

from pyspark.sql import functions as F

breakdown = spark.table(
    "workspace.ecom_poc.gold_revenue_breakdown"
)

# Check the table summary
summary = breakdown.agg(
    F.count("*").alias("gold_rows"),
    F.sum("total_orders").alias("orders_represented"),
    F.sum("completed_orders").alias("completed_orders")
).first()

# Verify uniqueness at the declared grain
duplicate_groups = (
    breakdown
    .groupBy(
        "sales_date",
        "category",
        "country",
        "currency"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Revenue breakdown rows:", summary["gold_rows"])
print("Orders represented:", summary["orders_represented"])
print("Completed orders:", summary["completed_orders"])
print("Duplicate groups:", duplicate_groups)

assert summary["gold_rows"] > 0
assert summary["orders_represented"] == 984_082
assert summary["completed_orders"] == 247_641
assert duplicate_groups == 0

print("REVENUE BREAKDOWN VERIFICATION: PASSED")


In [0]:

from pyspark.sql import functions as F

# 1. Revenue from the daily sales table
daily = spark.table(
    "workspace.ecom_poc.gold_daily_sales"
).select(
    "sales_date",
    "currency",
    F.col("total_revenue").alias("daily_revenue")
)

# 2. Aggregate the detailed breakdown to daily grain
breakdown_daily = (
    spark.table(
        "workspace.ecom_poc.gold_revenue_breakdown"
    )
    .groupBy("sales_date", "currency")
    .agg(
        F.sum("total_revenue").alias("breakdown_revenue")
    )
)

# 3. Match the two datasets
comparison = (
    daily.alias("d")
    .join(
        breakdown_daily.alias("b"),
        on=["sales_date", "currency"],
        how="full_outer"
    )
)

# 4. Detect differences or missing groups
mismatches = comparison.filter(
    F.col("daily_revenue").isNull()
    | F.col("breakdown_revenue").isNull()
    | (
        F.col("daily_revenue")
        != F.col("breakdown_revenue")
    )
)

mismatch_count = mismatches.count()

print("Date/currency groups compared:", comparison.count())
print("Revenue mismatches:", mismatch_count)

assert mismatch_count == 0

print("GOLD REVENUE RECONCILIATION: PASSED")


In [0]:
display(
    spark.sql("""
        SELECT *
        FROM workspace.ecom_poc.gold_revenue_breakdown
        ORDER BY sales_date, country, category
        LIMIT 20
    """)
)

In [0]:

spark.sql("""
CREATE OR REPLACE TABLE workspace.ecom_poc.gold_payment_success
USING DELTA
AS
SELECT
    CAST(txn_ts AS DATE) AS txn_date,
    payment_method,
    currency,

    COUNT(*) AS total_transactions,

    COUNT_IF(txn_status = 'SUCCESS') AS successful_transactions,

    COUNT_IF(txn_status = 'FAILED') AS failed_transactions,

    COUNT_IF(txn_status = 'PENDING') AS pending_transactions,

    COUNT_IF(txn_status = 'REFUNDED') AS refunded_transactions,

    ROUND(
        100.0 * COUNT_IF(txn_status = 'SUCCESS')
        / COUNT(*),
        2
    ) AS success_rate_pct

FROM workspace.ecom_poc.silver_transactions

GROUP BY
    CAST(txn_ts AS DATE),
    payment_method,
    currency
""")

print("Gold payment success table created!")


In [0]:

from pyspark.sql import functions as F

payments = spark.table(
    "workspace.ecom_poc.gold_payment_success"
)

# Check totals and success-rate range
summary = payments.agg(
    F.count("*").alias("gold_rows"),
    F.sum("total_transactions").alias("total"),
    F.sum("successful_transactions").alias("successful"),
    F.sum("failed_transactions").alias("failed"),
    F.sum("pending_transactions").alias("pending"),
    F.sum("refunded_transactions").alias("refunded"),
    F.min("success_rate_pct").alias("min_rate"),
    F.max("success_rate_pct").alias("max_rate")
).first()

# Verify uniqueness of date/method/currency groups
duplicate_groups = (
    payments
    .groupBy("txn_date", "payment_method", "currency")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Verify the status counts inside every group
invalid_groups = payments.filter(
    F.col("total_transactions") != (
        F.col("successful_transactions")
        + F.col("failed_transactions")
        + F.col("pending_transactions")
        + F.col("refunded_transactions")
    )
).count()

print("Gold payment rows:", summary["gold_rows"])
print("Total transactions:", summary["total"])
print("Successful:", summary["successful"])
print("Failed:", summary["failed"])
print("Pending:", summary["pending"])
print("Refunded:", summary["refunded"])
print("Minimum success rate:", summary["min_rate"])
print("Maximum success rate:", summary["max_rate"])
print("Duplicate groups:", duplicate_groups)
print("Invalid status-count groups:", invalid_groups)

assert summary["total"] == 980_082
assert (
    summary["successful"]
    + summary["failed"]
    + summary["pending"]
    + summary["refunded"]
) == summary["total"]
assert duplicate_groups == 0
assert invalid_groups == 0
assert 0 <= summary["min_rate"] <= 100
assert 0 <= summary["max_rate"] <= 100

print("GOLD PAYMENT SUCCESS VERIFICATION: PASSED")


In [0]:

display(
    spark.sql("""
        SELECT *
        FROM workspace.ecom_poc.gold_payment_success
        ORDER BY txn_date, payment_method, currency
        LIMIT 20
    """)
)


In [0]:

spark.sql("""
CREATE OR REPLACE TABLE workspace.ecom_poc.gold_top_customers
USING DELTA
AS
SELECT
    order_date AS sales_date,
    customer_id,
    currency,

    COUNT(*) AS total_orders,

    COUNT_IF(
        order_status = 'COMPLETED'
    ) AS completed_orders,

    CAST(
        SUM(revenue) AS DECIMAL(28,2)
    ) AS total_revenue

FROM workspace.ecom_poc.gold_order_revenue_base

GROUP BY
    order_date,
    customer_id,
    currency
""")

print("Gold top customers table created!")


In [0]:

from pyspark.sql import functions as F

customers = spark.table(
    "workspace.ecom_poc.gold_top_customers"
)

summary = customers.agg(
    F.count("*").alias("gold_rows"),
    F.sum("total_orders").alias("orders_represented"),
    F.sum("completed_orders").alias("completed_orders")
).first()

duplicate_groups = (
    customers
    .groupBy("sales_date", "customer_id", "currency")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

negative_revenue = customers.filter(
    F.col("total_revenue") < 0
).count()

print("Gold customer rows:", summary["gold_rows"])
print("Orders represented:", summary["orders_represented"])
print("Completed orders:", summary["completed_orders"])
print("Duplicate groups:", duplicate_groups)
print("Negative revenue groups:", negative_revenue)

assert summary["gold_rows"] > 0
assert summary["orders_represented"] == 984_082
assert summary["completed_orders"] == 247_641
assert duplicate_groups == 0
assert negative_revenue == 0

print("GOLD CUSTOMER VERIFICATION: PASSED")


In [0]:

from pyspark.sql import functions as F

# Revenue calculated from customers
customer_daily = (
    spark.table("workspace.ecom_poc.gold_top_customers")
    .groupBy("sales_date", "currency")
    .agg(
        F.sum("total_revenue").alias("customer_revenue")
    )
)

# Revenue calculated from daily sales
daily_sales = (
    spark.table("workspace.ecom_poc.gold_daily_sales")
    .select(
        "sales_date",
        "currency",
        F.col("total_revenue").alias("daily_revenue")
    )
)

# Compare both sources
comparison = customer_daily.join(
    daily_sales,
    on=["sales_date", "currency"],
    how="full_outer"
)

mismatches = comparison.filter(
    F.col("customer_revenue").isNull()
    | F.col("daily_revenue").isNull()
    | (F.col("customer_revenue") != F.col("daily_revenue"))
)

mismatch_count = mismatches.count()

print("Date/currency groups compared:", comparison.count())
print("Revenue mismatches:", mismatch_count)

assert mismatch_count == 0
assert comparison.count() == 2400

print("CUSTOMER REVENUE RECONCILIATION: PASSED")


In [0]:

display(
    spark.sql("""
        WITH customer_totals AS (
            SELECT
                customer_id,
                currency,
                SUM(total_orders) AS total_orders,
                SUM(total_revenue) AS customer_revenue

            FROM workspace.ecom_poc.gold_top_customers

            GROUP BY customer_id, currency
        ),

        ranked_customers AS (
            SELECT
                *,
                ROW_NUMBER() OVER (
                    PARTITION BY currency
                    ORDER BY
                        customer_revenue DESC,
                        customer_id ASC
                ) AS customer_rank

            FROM customer_totals
        )

        SELECT *
        FROM ranked_customers
        WHERE customer_rank <= 10
        ORDER BY currency, customer_rank
    """)
)


In [0]:

from pyspark.sql import functions as F

tables = {
    "foundation": ("gold_order_revenue_base", "revenue"),
    "daily_sales": ("gold_daily_sales", "total_revenue"),
    "breakdown": ("gold_revenue_breakdown", "total_revenue"),
    "customers": ("gold_top_customers", "total_revenue")
}

results = []

for name, (table, column) in tables.items():
    df = spark.table(f"workspace.ecom_poc.{table}")

    totals = (
        df.groupBy("currency")
        .agg(F.sum(column).alias("revenue"))
        .withColumn("source", F.lit(name))
    )

    results.append(totals)

combined = results[0]

for df in results[1:]:
    combined = combined.unionByName(df)

# Verify all four sources agree within each currency
checks = (
    combined
    .groupBy("currency")
    .agg(
        F.count("*").alias("source_count"),
        F.countDistinct("revenue").alias("distinct_totals"),
        F.min("revenue").alias("min_revenue"),
        F.max("revenue").alias("max_revenue")
    )
)

display(checks)

rows = checks.collect()

assert len(rows) == 4

for row in rows:
    assert row["source_count"] == 4
    assert row["distinct_totals"] == 1
    assert row["min_revenue"] == row["max_revenue"]

print("FINAL GOLD REVENUE VERIFICATION: PASSED")
